## 1. Nạp tập dữ liệu sạch từ file CSV

In [87]:
import os
import re
import pandas as pd
import numpy as np

# 1. Đường dẫn file CSV sạch vừa lưu
csv_path = os.path.abspath('../../data/notebook/content_base/jobs_clean.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.abspath('data/notebook/content_base/jobs_clean.csv')

# 2. Đọc file CSV
jobs = pd.read_csv(csv_path)
print(f" Đã nạp thành công: {jobs.shape[0]:,} dòng x {jobs.shape[1]} cột")
print("Danh sách các cột:", list(jobs.columns))
jobs.head(5)


 Đã nạp thành công: 3,836 dòng x 16 cột
Danh sách các cột: ['job_id', 'company_name', 'title', 'description_clean', 'required_skills', 'preferred_skills', 'experience_min', 'experience_max', 'location', 'city', 'salary_min', 'salary_max', 'employment_type', 'work_mode', 'deadline', 'is_active']


,job_id,company_name,title,description_clean,required_skills,preferred_skills,experience_min,experience_max,location,city,salary_min,salary_max,employment_type,work_mode,deadline,is_active
0,topcv_2136013,1 THÀNH VIÊN CÔNG NGHỆ GIÁO DỤC SOTATEK,Kỹ Sư Hạ Tầng Công Nghệ Thông Tin,"1. Phụ trách lắp đặt, cấu hình và bảo trì các ...","It - Phần Mềm, Sản Xuất, Tiếng trung giao tiếp...",NaN,1.0,1.0,"KCN Phước Nam 2, Ninh Thuận cũ, Xã Thuận Nam, ...",Khác,NaN,40000000.0,Full-time,Onsite,2026-11-08,True
1,topcv_1986323,1 THÀNH VIÊN ISOFHCARE,Chuyên Viên BA Nghiệp Vụ,Hỗ trợ Phân tích Yêu cầu: Làm việc dưới sự hướ...,"Chuyên môn công nghệ thông tin khác, It - Phần...",NaN,3.0,3.0,"Tòa nhà One Mount. 458 Minh Khai, Phường Hai B...",Hà Nội,NaN,25000000.0,Full-time,Onsite,2026-11-25,True
2,topcv_2077790,1 THÀNH VIÊN ISOFHCARE,Nhân Viên Phân Tích Yêu Cầu Ứng Dụng (BA),"Business Analyst chịu trách nhiệm thu thập, ph...","Chuyên môn công nghệ thông tin khác, It - Phần...",NaN,3.0,3.0,"One Mount Building, 458 P. Minh Khai, Phường H...",Hà Nội,NaN,25000000.0,Full-time,Onsite,2026-10-22,False
3,topcv_2339964,1 THÀNH VIÊN TRUYỀN THÔNG YUM NETWORK,Nhân Viên Designer Thiết Kế Parttime,"Thiết kế, quản lý và phát triển về mặt mỹ thuậ...","Animation Design, Marketing, Quảng Cáo",NaN,1.0,1.0,"23 Duy Tân,Cầu Giấy, Phường Cầu Giấy, Hà Nội",Hà Nội,5000000.0,6000000.0,Part-time,Onsite,2026-11-23,True
4,topcv_2334350,1C,Chuyên Viên Kinh Doanh Phần Mềm Quản Trị,"Chủ động tìm kiếm, tiếp cận và phát triển khác...","Kinh Doanh Phần Mềm, Direct Sales, B2B, It - P...",NaN,2.0,2.0,"Tầng 21 Tòa Century - 458 đường Minh Khai, Tim...",Hà Nội,NaN,NaN,Full-time,Onsite,2026-11-18,True


## 2. Kiểm tra Cấu trúc dữ liệu & Kiểu dữ liệu (Data Types)

In [88]:
print("=== TỔNG QUAN KIỂU DỮ LIỆU & BỘ NHỚ ===")
jobs.info()


=== TỔNG QUAN KIỂU DỮ LIỆU & BỘ NHỚ ===
<class 'pandas.DataFrame'>
RangeIndex: 3836 entries, 0 to 3835
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_id             3836 non-null   str    
 1   company_name       3836 non-null   str    
 2   title              3836 non-null   str    
 3   description_clean  3836 non-null   str    
 4   required_skills    3816 non-null   str    
 5   preferred_skills   0 non-null      float64
 6   experience_min     3836 non-null   float64
 7   experience_max     3836 non-null   float64
 8   location           3836 non-null   str    
 9   city               3836 non-null   str    
 10  salary_min         1495 non-null   float64
 11  salary_max         1713 non-null   float64
 12  employment_type    3836 non-null   str    
 13  work_mode          3836 non-null   str    
 14  deadline           3836 non-null   str    
 15  is_active          3836 non-null   bool   


## 3. Kiểm tra Độ đầy đủ dữ liệu (Completeness & Missing Rate)

In [89]:
total_rows = len(jobs)
missing_stats = []

for col in jobs.columns:
    null_count = jobs[col].isnull().sum()
    # Đếm thêm chuỗi rỗng nếu là kiểu object/string
    empty_count = (jobs[col].astype(str).str.strip() == '').sum() if jobs[col].dtype == 'object' else 0
    total_missing = null_count + empty_count
    missing_pct = (total_missing / total_rows) * 100
    valid_pct = 100.0 - missing_pct
    
    missing_stats.append({
        'Cột': col,
        'Số dòng thiếu': total_missing,
        'Tỷ lệ thiếu (%)': round(missing_pct, 2),
        'Độ đầy đủ (%)': round(valid_pct, 2)
    })

df_missing = pd.DataFrame(missing_stats).sort_values(by='Độ đầy đủ (%)', ascending=False)
display(df_missing)


,Cột,Số dòng thiếu,Tỷ lệ thiếu (%),Độ đầy đủ (%)
0,job_id,0,0.00,100.00
1,company_name,0,0.00,100.00
2,title,0,0.00,100.00
3,description_clean,0,0.00,100.00
7,experience_max,0,0.00,100.00
6,experience_min,0,0.00,100.00
9,city,0,0.00,100.00
8,location,0,0.00,100.00
14,deadline,0,0.00,100.00
15,is_active,0,0.00,100.00


## 4. Thống kê mô tả các trường số (Mức lương & Kinh nghiệm)

Quy đổi cột lương sang đơn vị **triệu VNĐ/tháng** để dễ quan sát phân bố mức lương tối thiểu và tối đa.

In [90]:
# Thống kê mô tả số năm kinh nghiệm
print("=== PHÂN BỐ SỐ NĂM KINH NGHIỆM (NĂM) ===")
display(jobs[['experience_min', 'experience_max']].describe().round(2))

# Thống kê mức lương theo đơn vị triệu VNĐ/tháng
print("\n=== PHÂN BỐ MỨC LƯƠNG (TRIỆU VNĐ/THÁNG) ===")
salary_summary = pd.DataFrame({
    'Lương tối thiểu (triệu VNĐ)': (jobs['salary_min'] / 1_000_000).describe(),
    'Lương tối đa (triệu VNĐ)': (jobs['salary_max'] / 1_000_000).describe()
}).round(2)
display(salary_summary)


=== PHÂN BỐ SỐ NĂM KINH NGHIỆM (NĂM) ===


,experience_min,experience_max
count,3836.00,3836.00
mean,2.36,2.36
std,2.00,2.00
min,0.00,0.00
25%,1.00,1.00
50%,2.00,2.00
75%,3.00,3.00
max,20.00,20.00



=== PHÂN BỐ MỨC LƯƠNG (TRIỆU VNĐ/THÁNG) ===


,Lương tối thiểu (triệu VNĐ),Lương tối đa (triệu VNĐ)
count,1495.00,1713.00
mean,22762.60,33885.79
std,631197.73,1059567.30
min,0.00,0.00
25%,10.00,10.00
50%,14.00,16.00
75%,20.00,30.00
max,20000000.00,40000000.00


## 5. Phân bố các trường phân loại (Categorical Distribution)

In [91]:
# 1. Phân bố theo thành phố
print("=== 1. PHÂN BỐ THEO THÀNH PHỐ (CITY) ===")
city_df = jobs['city'].value_counts(dropna=False).reset_index()
city_df.columns = ['Thành phố', 'Số lượng']
city_df['Tỷ lệ (%)'] = (city_df['Số lượng'] / len(jobs) * 100).round(2)
display(city_df)

# 2. Phân bố theo mô hình làm việc
print("\n=== 2. PHÂN BỐ THEO MÔ HÌNH LÀM VIỆC (WORK_MODE) ===")
wm_df = jobs['work_mode'].value_counts(dropna=False).reset_index()
wm_df.columns = ['Work Mode', 'Số lượng']
wm_df['Tỷ lệ (%)'] = (wm_df['Số lượng'] / len(jobs) * 100).round(2)
display(wm_df)

# 3. Phân bố theo hình thức làm việc
print("\n=== 3. PHÂN BỐ THEO HÌNH THỨC LÀM VIỆC (EMPLOYMENT_TYPE) ===")
emp_df = jobs['employment_type'].value_counts(dropna=False).reset_index()
emp_df.columns = ['Employment Type', 'Số lượng']
emp_df['Tỷ lệ (%)'] = (emp_df['Số lượng'] / len(jobs) * 100).round(2)
display(emp_df)

# 4. Phân bố theo trạng thái hiệu lực
print("\n=== 4. TRẠNG THÁI HIỆU LỰC (IS_ACTIVE) ===")
act_df = jobs['is_active'].value_counts(dropna=False).reset_index()
act_df.columns = ['Is Active', 'Số lượng']
act_df['Tỷ lệ (%)'] = (act_df['Số lượng'] / len(jobs) * 100).round(2)
display(act_df)


=== 1. PHÂN BỐ THEO THÀNH PHỐ (CITY) ===


,Thành phố,Số lượng,Tỷ lệ (%)
0,Hà Nội,2031,52.95
1,Hồ Chí Minh,1053,27.45
2,Khác,689,17.96
3,Đà Nẵng,63,1.64



=== 2. PHÂN BỐ THEO MÔ HÌNH LÀM VIỆC (WORK_MODE) ===


,Work Mode,Số lượng,Tỷ lệ (%)
0,Onsite,3789,98.77
1,Remote,32,0.83
2,Hybrid,15,0.39



=== 3. PHÂN BỐ THEO HÌNH THỨC LÀM VIỆC (EMPLOYMENT_TYPE) ===


,Employment Type,Số lượng,Tỷ lệ (%)
0,Full-time,3659,95.39
1,Intern,158,4.12
2,Contract,15,0.39
3,Part-time,4,0.10



=== 4. TRẠNG THÁI HIỆU LỰC (IS_ACTIVE) ===


,Is Active,Số lượng,Tỷ lệ (%)
0,True,3035,79.12
1,False,801,20.88


## 6. Phân tích đặc trưng văn bản & Top Kỹ năng yêu cầu

In [92]:
# 1. Thống kê độ dài văn bản mô tả công việc (description_clean)
desc_lengths = jobs['description_clean'].dropna().str.len()
print("=== ĐỘ DÀI MÔ TẢ CÔNG VIỆC (SỐ KÝ TỰ) ===")
display(desc_lengths.describe().round(0))

# 2. Bóc tách và đếm tần suất Top 20 kỹ năng yêu cầu
from collections import Counter

skill_counter = Counter()
for text in jobs['required_skills'].dropna():
    skills = [s.strip() for s in str(text).split(',') if s.strip()]
    skill_counter.update(skills)

top_skills_df = pd.DataFrame(skill_counter.most_common(20), columns=['Kỹ năng', 'Số lần xuất hiện'])
top_skills_df['Tỷ lệ (%)'] = (top_skills_df['Số lần xuất hiện'] / len(jobs) * 100).round(2)

print("\n=== TOP 20 KỸ NĂNG XUẤT HIỆN NHIỀU NHẤT TRONG DATASET ===")
display(top_skills_df)


=== ĐỘ DÀI MÔ TẢ CÔNG VIỆC (SỐ KÝ TỰ) ===


count     3836.0
mean      2357.0
std       1171.0
min        340.0
25%       1568.0
50%       2122.0
75%       2839.0
max      12642.0
Name: description_clean, dtype: float64


=== TOP 20 KỸ NĂNG XUẤT HIỆN NHIỀU NHẤT TRONG DATASET ===


,Kỹ năng,Số lần xuất hiện,Tỷ lệ (%)
0,It - Phần Mềm,1085,28.28
1,SQL,534,13.92
2,Python,443,11.55
3,Thiết kế đồ họa (graphic design),381,9.93
4,B2B,322,8.39
5,CI/CD,269,7.01
6,Java,262,6.83
7,Marketing,260,6.78
8,Excel,257,6.70
9,Quảng Cáo,254,6.62


## 6. Xây dựng "Content Soup" (Tổ hợp đặc trưng có gán trọng số)

Để ưu tiên các trường quan trọng nhất trong việc ra quyết định tuyển dụng:
- **Chức danh (`title`)**: Trọng số x3 (xác định định hướng vị trí công việc).
- **Kỹ năng cốt lõi (`required_skills`)**: Trọng số x4 (yếu tố cốt lõi quyết định độ khớp).
- **Mô tả chi tiết (`description_clean`)**: Trọng số x1 (cung cấp thêm ngữ cảnh về dự án và công nghệ).

In [93]:
def build_content_soup(row):
    title = str(row.get('title', '')) if pd.notnull(row.get('title')) else ''
    skills = str(row.get('required_skills', '')) if pd.notnull(row.get('required_skills')) else ''
    desc = str(row.get('description_clean', '')) if pd.notnull(row.get('description_clean')) else ''
    
    # Nhân trọng số bằng cách lặp lại từ khóa
    soup = f"{title} {title} {title} {skills} {skills} {skills} {skills} {desc}"
    return soup.strip()

jobs['content_soup'] = jobs.apply(build_content_soup, axis=1)

print("=== MẪU CONTENT SOUP CỦA 2 CÔNG VIỆC ĐẦU TIÊN ===")
for idx, s in enumerate(jobs['content_soup'].head(2)):
    print(f"\n[Job {idx+1}] {jobs.iloc[idx]['title']}:")
    print(s[:300] + "...")


=== MẪU CONTENT SOUP CỦA 2 CÔNG VIỆC ĐẦU TIÊN ===

[Job 1] Kỹ Sư Hạ Tầng Công Nghệ Thông Tin:
Kỹ Sư Hạ Tầng Công Nghệ Thông Tin Kỹ Sư Hạ Tầng Công Nghệ Thông Tin Kỹ Sư Hạ Tầng Công Nghệ Thông Tin It - Phần Mềm, Sản Xuất, Tiếng trung giao tiếp thành thạo, Tiếng Trung Hsk 4, Linux It - Phần Mềm, Sản Xuất, Tiếng trung giao tiếp thành thạo, Tiếng Trung Hsk 4, Linux It - Phần Mềm, Sản Xuất, Tiếng...

[Job 2] Chuyên Viên BA Nghiệp Vụ:
Chuyên Viên BA Nghiệp Vụ Chuyên Viên BA Nghiệp Vụ Chuyên Viên BA Nghiệp Vụ Chuyên môn công nghệ thông tin khác, It - Phần Mềm, Y Tế, Dược Phẩm, SQL, Jira Chuyên môn công nghệ thông tin khác, It - Phần Mềm, Y Tế, Dược Phẩm, SQL, Jira Chuyên môn công nghệ thông tin khác, It - Phần Mềm, Y Tế, Dược Phẩm...


## 7. Vector hóa với TF-IDF & Tính Ma trận Cosine Similarity

- Sử dụng `TfidfVectorizer` với `ngram_range=(1, 2)` để nắm bắt cả từ đơn (`python`, `sql`) và cụm từ ghép (`spring boot`, `machine learning`, `data analyst`).
- Giới hạn `max_features=10,000` và `min_df=2` để lọc bớt từ hiếm xuất hiện dưới 2 lần, tối ưu bộ nhớ.

In [94]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

start_time = time.time()

# 1. Khởi tạo và fit-transform TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)
tfidf_matrix = tfidf.fit_transform(jobs['content_soup'])

# 2. Tính ma trận Cosine Similarity giữa tất cả các công việc (Job-to-Job)
cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

elapsed = time.time() - start_time
print(f" Đã vector hóa thành công {tfidf_matrix.shape[0]:,} jobs x {tfidf_matrix.shape[1]:,} features")
print(f" Ma trận Cosine Similarity: {cosine_sim_matrix.shape} (Dung lượng: ~{cosine_sim_matrix.nbytes / 1024**2:.1f} MB)")
print(f"⏱ Thời gian thực thi: {elapsed:.2f} giây")


 Đã vector hóa thành công 3,836 jobs x 10,000 features
 Ma trận Cosine Similarity: (3836, 3836) (Dung lượng: ~112.3 MB)
⏱ Thời gian thực thi: 4.47 giây


## 9. Hệ thống Gợi ý Việc làm Toàn diện theo Hồ sơ & CV Ứng viên (Unified Recommender Engine)

Hệ thống hợp nhất linh hoạt cho phép ứng viên cung cấp thông tin theo nhiều cách:
1. **Tự chọn kỹ năng & Chức danh** (nhập danh sách `skills`).
2. **Dán đoạn văn bản CV / Tóm tắt năng lực** (`cv_text`): Hệ thống tự động bóc tách kỹ năng công nghệ (`extract_skills_from_text`).
3. **Kết hợp cả hai**: Dán CV + tích chọn thêm kỹ năng + đặt bộ lọc lương tối thiểu (`min_salary`), địa điểm (`preferred_city`), kinh nghiệm (`user_exp`).
4. **Phân tích khoảng cách kỹ năng (Skill Gap Analysis)**: Chỉ rõ những gì bạn đã đáp ứng (Điểm mạnh) và những gì công ty cần mà bạn còn thiếu (Gap cần học thêm).
5. **So khớp chuyên sâu 1:1 (`analyze_cv_fit_for_job`)**: Báo cáo phân tích chi tiết hồ sơ với một công việc cụ thể.

In [95]:
import re

# Danh bạ quy chuẩn tên công nghệ IT phục vụ nhận diện tự động từ văn bản
CANONICAL_SKILL_MAP = {
    'python': 'Python', 'java': 'Java', 'javascript': 'JavaScript', 'js': 'JavaScript',
    'typescript': 'TypeScript', 'ts': 'TypeScript', 'c++': 'C++', 'c#': 'C#',
    'php': 'PHP', 'golang': 'Golang', 'go': 'Golang', 'ruby': 'Ruby',
    'react': 'React', 'reactjs': 'React', 'vue': 'Vue', 'vuejs': 'Vue',
    'angular': 'Angular', 'node': 'Node.js', 'nodejs': 'Node.js',
    'spring boot': 'Spring Boot', 'springboot': 'Spring Boot', 'laravel': 'Laravel',
    '.net': '.NET', 'dotnet': '.NET', 'sql': 'SQL', 'mysql': 'MySQL',
    'postgresql': 'PostgreSQL', 'postgres': 'PostgreSQL', 'mongodb': 'MongoDB',
    'redis': 'Redis', 'kafka': 'Kafka', 'elasticsearch': 'Elasticsearch',
    'docker': 'Docker', 'kubernetes': 'Kubernetes', 'k8s': 'Kubernetes',
    'aws': 'AWS', 'azure': 'Azure', 'gcp': 'GCP', 'ci/cd': 'CI/CD', 'cicd': 'CI/CD',
    'git': 'Git', 'linux': 'Linux', 'figma': 'Figma', 'power bi': 'Power BI',
    'tableau': 'Tableau', 'excel': 'Excel', 'jira': 'Jira'
}

def extract_skills_from_text(text):
    """Tự động nhận diện và bóc tách các kỹ năng IT từ bất kỳ đoạn văn bản nào."""
    text_lower = str(text).lower()
    detected = []
    seen = set()
    for key, val in CANONICAL_SKILL_MAP.items():
        if re.search(r'\b' + re.escape(key) + r'\b', text_lower):
            if val.lower() not in seen:
                seen.add(val.lower())
                detected.append(val)
    return detected

def recommend_jobs(target_title="", cv_text="", skills=None, user_exp=1.0, preferred_city="Tất cả", min_salary=None, top_n=10):
    """
    Hàm gợi ý việc làm toàn diện kết hợp: Vị trí + Kỹ năng chọn tay + Đoạn văn bản CV + Bộ lọc kinh nghiệm/lương/địa điểm.
    """
    manual_skills = skills or []
    extracted_skills = extract_skills_from_text(cv_text) if cv_text else []
    
    # 1. Hợp nhất kỹ năng (loại bỏ trùng lặp)
    all_skills = []
    seen_skills = set()
    for s in manual_skills + extracted_skills:
        if s.lower() not in seen_skills:
            seen_skills.add(s.lower())
            all_skills.append(s)
            
    skills_soup = " ".join(all_skills)
    
    # 2. Xây dựng Profile Soup có gán trọng số
    parts = []
    if target_title:
        parts.extend([target_title, target_title, target_title])
    if skills_soup:
        parts.extend([skills_soup, skills_soup, skills_soup, skills_soup])
    if cv_text:
        parts.append(cv_text)
        
    profile_soup = " ".join(parts).strip()
    if not profile_soup:
        print(" Vui lòng cung cấp ít nhất một thông tin: Vị trí (target_title), Kỹ năng (skills), hoặc Văn bản CV (cv_text).")
        return pd.DataFrame()
        
    # 3. Vectorize và tính Cosine Similarity (Stage 1)
    profile_vec = tfidf.transform([profile_soup])
    scores = cosine_similarity(profile_vec, tfidf_matrix).flatten()
    
    # 4. Lọc nghiệp vụ & Phân tích khoảng cách kỹ năng (Stage 2)
    results = []
    for idx, score in enumerate(scores):
        row = jobs.iloc[idx]
        if 'is_active' in row and not row['is_active']:
            continue
        if preferred_city != 'Tất cả':
            if row['city'] != preferred_city and row['work_mode'] != 'Remote':
                continue
        job_min_exp = row['experience_min']
        if pd.notnull(job_min_exp) and job_min_exp > (user_exp + 1.0):
            continue
        if min_salary is not None and pd.notnull(row['salary_max']) and row['salary_max'] < min_salary:
            continue
            
        job_skills = [s.strip() for s in str(row['required_skills']).split(',') if s.strip()]
        job_skills_lower = [s.lower() for s in job_skills]
        
        matched = [s for s, sl in zip(job_skills, job_skills_lower) if sl in seen_skills]
        missing = [s for s, sl in zip(job_skills, job_skills_lower) if sl not in seen_skills]
        coverage_pct = round(len(matched) / len(job_skills) * 100, 1) if job_skills else 100.0
        
        results.append({
            'Tiêu đề': row['title'],
            'Công ty': row['company_name'],
            'Độ phù hợp (%)': round(score * 100, 1),
            'Độ phủ kỹ năng (%)': f"{coverage_pct}% ({len(matched)}/{len(job_skills)})",
            'Kỹ năng đã có (Điểm mạnh)': ', '.join(matched[:4]) if matched else 'Khớp theo vị trí/mô tả',
            'Kỹ năng còn thiếu (Gap)': ', '.join(missing[:4]) if missing else 'Đầy đủ',
            'Kinh nghiệm yêu cầu': f"{row['experience_min']} năm" if pd.notnull(row['experience_min']) else 'Không yêu cầu',
            'Thành phố': row['city'],
            'Mức lương': f"{row['salary_min']/1e6:.0f} - {row['salary_max']/1e6:.0f} tr" if pd.notnull(row['salary_min']) and pd.notnull(row['salary_max']) else ('Lên đến ' + f"{row['salary_max']/1e6:.0f} tr" if pd.notnull(row['salary_max']) else 'Thoả thuận')
        })
        
    df_res = pd.DataFrame(results).sort_values(by='Độ phù hợp (%)', ascending=False)
    return df_res.head(top_n)

def analyze_cv_fit_for_job(job_identifier, cv_text="", skills=None, user_exp=1.0):
    """
    Báo cáo đánh giá chi tiết 1:1 giữa hồ sơ ứng viên và một công việc cụ thể.
    """
    if isinstance(job_identifier, (int, np.integer)):
        idx = int(job_identifier)
    elif isinstance(job_identifier, str):
        match = jobs[jobs['job_id'] == job_identifier]
        if match.empty:
            match = jobs[jobs['title'].str.contains(job_identifier, case=False, na=False, regex=False)]
        if match.empty:
            return f" Không tìm thấy công việc: {job_identifier}"
        idx = match.index[0]
        
    job = jobs.iloc[idx]
    
    manual_skills = skills or []
    extracted_skills = extract_skills_from_text(cv_text) if cv_text else []
    all_user_skills = []
    seen_skills = set()
    for s in manual_skills + extracted_skills:
        if s.lower() not in seen_skills:
            seen_skills.add(s.lower())
            all_user_skills.append(s)
            
    input_text = f"{' '.join(all_user_skills)} {cv_text}".strip()
    user_vec = tfidf.transform([input_text])
    job_vec = tfidf.transform([job['content_soup']])
    sim_score = cosine_similarity(user_vec, job_vec)[0][0]
    
    job_skills = [s.strip() for s in str(job['required_skills']).split(',') if s.strip()]
    job_skills_lower = [s.lower() for s in job_skills]
    
    matched = [s for s, sl in zip(job_skills, job_skills_lower) if sl in seen_skills]
    missing = [s for s, sl in zip(job_skills, job_skills_lower) if sl not in seen_skills]
    
    exp_required = job['experience_min'] if pd.notnull(job['experience_min']) else 0.0
    if user_exp >= exp_required:
        exp_status = f" Đạt yêu cầu ({user_exp} năm >= {exp_required} năm)"
    elif user_exp + 1.0 >= exp_required:
        exp_status = f" Chấp nhận được ({user_exp} năm xấp xỉ yêu cầu {exp_required} năm)"
    else:
        exp_status = f" Chưa đủ kinh nghiệm ({user_exp} năm < yêu cầu {exp_required} năm)"
        
    coverage_pct = round(len(matched) / len(job_skills) * 100, 1) if job_skills else 100.0
    
    report = f"""================================================================================
 BÁO CÁO ĐÁNH GIÁ ĐỘ PHÙ HỢP HỒ SƠ VỚI CÔNG VIỆC (FIT REPORT)
================================================================================
Công việc mục tiêu: {job['title']}
Công ty:             {job['company_name']}
Địa điểm:            {job['city']} ({job['work_mode']})
Mức lương:           {f"{job['salary_min']/1e6:.0f} - {job['salary_max']/1e6:.0f} triệu" if pd.notnull(job['salary_min']) and pd.notnull(job['salary_max']) else ('Lên đến ' + f"{job['salary_max']/1e6:.0f} triệu" if pd.notnull(job['salary_max']) else 'Thoả thuận')}

--- 1. ĐIỂM SỐ TỔNG QUAN ---
• Điểm tương đồng nội dung (Content Similarity): {sim_score * 100:.1f}%
• Tỷ lệ đáp ứng kỹ năng (Skill Coverage):         {coverage_pct}% ({len(matched)}/{len(job_skills)} kỹ năng)
• Đánh giá số năm kinh nghiệm:                   {exp_status}

--- 2. CHI TIẾT KỸ NĂNG ---
 Kỹ năng bạn đã có ({len(matched)}):
   {', '.join(matched) if matched else 'Không có kỹ năng trùng khớp trực tiếp'}
 Kỹ năng còn thiếu mà công ty yêu cầu ({len(missing)}):
   {', '.join(missing) if missing else 'Tuyệt vời! Bạn đáp ứng đầy đủ tất cả kỹ năng cốt lõi'}

--- 3. LỜI KHUYÊN DÀNH CHO BẠN ---
"""
    if missing:
        report += f" Để tăng cơ hội trúng tuyển, bạn nên bổ sung thêm kinh nghiệm hoặc học thêm về: {', '.join(missing)}.\n"
    else:
        report += " Hồ sơ của bạn rất mạnh cho vị trí này! Hãy tự tin nộp đơn ứng tuyển ngay.\n"
    report += "================================================================================"
    return report

print(" Đã khởi tạo thành công Unified Recommender Engine:")
print("- extract_skills_from_text(text)")
print("- recommend_jobs(target_title, cv_text, skills, user_exp, preferred_city, min_salary, top_n)")
print("- analyze_cv_fit_for_job(job_identifier, cv_text, skills, user_exp)")


 Đã khởi tạo thành công Unified Recommender Engine:
- extract_skills_from_text(text)
- recommend_jobs(target_title, cv_text, skills, user_exp, preferred_city, min_salary, top_n)
- analyze_cv_fit_for_job(job_identifier, cv_text, skills, user_exp)


## 10. Thử nghiệm Thực tế với các Kịch bản Đa dạng (Case Studies)

In [96]:
# --- Case 1: Ứng viên chỉ chọn kỹ năng và vị trí mong muốn ---
print("=== CASE 1: ỨNG VIÊN CHỈ CHỌN KỸ NĂNG BẰNG TAY (Backend Python, Hà Nội) ===")
res1 = recommend_jobs(
    target_title="Backend Developer Python",
    skills=["Python", "SQL", "Docker", "PostgreSQL"],
    user_exp=1.0,
    preferred_city="Hà Nội",
    top_n=5
)
display(res1)

# --- Case 2: Ứng viên dán đoạn mô tả kinh nghiệm / CV ---
print("\n=== CASE 2: ỨNG VIÊN DÁN VĂN BẢN CV (DevOps / Microservices, Hồ Chí Minh) ===")
cv_devops = """
Kỹ sư DevOps với 3 năm kinh nghiệm làm việc với hệ thống Microservices.
Thành thạo Docker, Kubernetes (K8s), xây dựng CI/CD pipeline tự động hóa với GitLab CI và Jenkins.
Kinh nghiệm quản trị hệ thống Linux, triển khai hạ tầng đám mây trên AWS (EC2, S3, RDS, EKS).
"""
res2 = recommend_jobs(
    target_title="DevOps Engineer",
    cv_text=cv_devops,
    user_exp=3.0,
    preferred_city="Hồ Chí Minh",
    top_n=5
)
display(res2)

# --- Case 3: Kết hợp cả CV + Kỹ năng chọn thêm + Lọc lương tối thiểu ---
print("\n=== CASE 3: KẾT HỢP CV + KỸ NĂNG BỔ SUNG + LỌC LƯƠNG TỐI THIỂU >= 12 TRIỆU ===")
cv_da = """
Chuyên viên phân tích dữ liệu với 1 năm kinh nghiệm.
Sử dụng thành thạo SQL để truy vấn và xử lý dữ liệu lớn, thành thạo Excel và Power BI xây dựng Dashboard quản trị kinh doanh.
"""
res3 = recommend_jobs(
    target_title="Data Analyst",
    cv_text=cv_da,
    skills=["Python", "Tableau"],
    user_exp=1.0,
    preferred_city="Tất cả",
    min_salary=12000000,
    top_n=5
)
display(res3)

# --- Case 4: So khớp chuyên sâu 1:1 giữa CV DevOps và công việc gợi ý Top 1 ---
top1_job = res2.iloc[0]['Tiêu đề']
rep4 = analyze_cv_fit_for_job(job_identifier=top1_job, cv_text=cv_devops, user_exp=3.0)
print("\n" + rep4)


=== CASE 1: ỨNG VIÊN CHỈ CHỌN KỸ NĂNG BẰNG TAY (Backend Python, Hà Nội) ===


,Tiêu đề,Công ty,Độ phù hợp (%),Độ phủ kỹ năng (%),Kỹ năng đã có (Điểm mạnh),Kỹ năng còn thiếu (Gap),Kinh nghiệm yêu cầu,Thành phố,Mức lương
876,Senior Backend Developer (Python),VODAPLAY,26.9,26.7% (4/15),"Python, SQL, PostgreSQL, Docker","Backend Developer, An Ninh Mạng, It - Phần Mềm...",2.0 năm,Hà Nội,Thoả thuận
351,IT System (Lead) - Onboard Sớm - Part Time,Giải pháp Phân tích dữ liệu Insight Data,24.6,33.3% (3/9),"Python, SQL, Docker","It Helpdesk, It Support, It - Phần Mềm, Tiếng ...",1.0 năm,Hà Nội,Thoả thuận
720,AI Engineer - Chuyên Viên Triển Khai & Vận Hàn...,THỰC PHẨM KIM PHÚ,20.8,42.9% (3/7),"Python, SQL, Docker","Thương Mại Điện Tử, Bán lẻ - hàng tiêu dùng - ...",2.0 năm,Hà Nội,Thoả thuận
969,Backend Developer,ĐẦU TƯ QUỐC TẾ DGLOBAL,20.4,20.0% (3/15),"SQL, PostgreSQL, Docker","Backend Developer, Java, Spring Boot, MySQL",2.0 năm,Hà Nội,Thoả thuận
899,Lập Trình Viên Backend / Web Developer (NodeJS...,Viện nghiên cứu và phát triển Sản phẩm thiên n...,20.2,16.7% (1/6),Docker,"Backend Developer, It - Phần Mềm, JavaScript, ...",0.0 năm,Hà Nội,10 - 10 tr



=== CASE 2: ỨNG VIÊN DÁN VĂN BẢN CV (DevOps / Microservices, Hồ Chí Minh) ===


,Tiêu đề,Công ty,Độ phù hợp (%),Độ phủ kỹ năng (%),Kỹ năng đã có (Điểm mạnh),Kỹ năng còn thiếu (Gap),Kinh nghiệm yêu cầu,Thành phố,Mức lương
362,Devops Engineer,IOTLINK,55.0,35.7% (5/14),"Docker, Kubernetes, AWS, CI/CD","Devops Engineer, Python, JavaScript, PHP",1.0 năm,Hồ Chí Minh,20 - 20 tr
689,System Engineer (Data Center),Viễn Thông Quốc Tế FPT,38.6,45.5% (5/11),"Docker, Kubernetes, AWS, CI/CD","System Engineer, Điện toán đám mây (cloud), Ti...",2.0 năm,Hồ Chí Minh,Thoả thuận
93,Senior DevOps Engineer,Chứng khoán HD,38.4,25.0% (4/16),"Docker, Kubernetes, CI/CD, Linux","Devops Engineer, Chứng Khoán, Tài Chính, Ngân ...",4.0 năm,Hồ Chí Minh,Thoả thuận
220,DevOps,DƯỢC PHẨM FPT LONG CHÂU,34.4,25.0% (3/12),"Docker, Kubernetes, AWS","Devops Engineer, It - Phần Mềm, Công Nghệ Kỹ T...",2.0 năm,Hồ Chí Minh,Thoả thuận
557,NodeJS Backend (TeamLead),TECHVIFY SOFTWARE.,33.7,33.3% (4/12),"Docker, Kubernetes, AWS, CI/CD","Backend Developer, It - Phần Mềm, Node.js, MySQL",2.0 năm,Hồ Chí Minh,Thoả thuận



=== CASE 3: KẾT HỢP CV + KỸ NĂNG BỔ SUNG + LỌC LƯƠNG TỐI THIỂU >= 12 TRIỆU ===


,Tiêu đề,Công ty,Độ phù hợp (%),Độ phủ kỹ năng (%),Kỹ năng đã có (Điểm mạnh),Kỹ năng còn thiếu (Gap),Kinh nghiệm yêu cầu,Thành phố,Mức lương
921,Chuyên Viên Phân Tích Dữ Liệu,Nhựa Tái Chế Duy Tân,48.0,42.9% (3/7),"SQL, Power BI, Excel","System administration, Technical Support, Data...",2.0 năm,Khác,13 - 15 tr
923,Nhân Viên Phân Tích Giữ Liệu & Dashboard Tại G...,Nông Nghiệp BaF,47.6,42.9% (3/7),"SQL, Power BI, Excel","It - Phần Mềm, Xây Dựng, Giáo Dục, Đào Tạo",1.0 năm,Hà Nội,22 - 27 tr
561,Senior Data Analyst - Green Delivery Project,GSM,47.1,66.7% (4/6),"SQL, Power BI, Python, Tableau","Tableu, Powerpoint",0.0 năm,Khác,20 - 38 tr
714,Middle Data Analyst (Từ 2 Năm Kinh Nghiệm),Khoa học Dữ liệu,46.6,66.7% (4/6),"Python, SQL, Power BI, Excel","It - Phần Mềm, Thương Mại Điện Tử",2.0 năm,Hà Nội,15 - 15 tr
523,Data Analyst (Mảng Tài Chính),GALAXY DEBT TRADING COMPANY LIMITED,45.3,66.7% (4/6),"Python, SQL, Power BI, Excel","Tài Chính, Ngân Hàng",1.0 năm,Hồ Chí Minh,15 - 15 tr



 BÁO CÁO ĐÁNH GIÁ ĐỘ PHÙ HỢP HỒ SƠ VỚI CÔNG VIỆC (FIT REPORT)
Công việc mục tiêu: Senior Devops Engineer
Công ty:             Aggregatori Capaci
Địa điểm:            Hà Nội (Onsite)
Mức lương:           Thoả thuận

--- 1. ĐIỂM SỐ TỔNG QUAN ---
• Điểm tương đồng nội dung (Content Similarity): 29.1%
• Tỷ lệ đáp ứng kỹ năng (Skill Coverage):         62.5% (5/8 kỹ năng)
• Đánh giá số năm kinh nghiệm:                    Chấp nhận được (3.0 năm xấp xỉ yêu cầu 4.0 năm)

--- 2. CHI TIẾT KỸ NĂNG ---
 Kỹ năng bạn đã có (5):
   Docker, Kubernetes, AWS, CI/CD, Linux
 Kỹ năng còn thiếu mà công ty yêu cầu (3):
   Devops Engineer, It - Phần Mềm, GCP

--- 3. LỜI KHUYÊN DÀNH CHO BẠN ---
 Để tăng cơ hội trúng tuyển, bạn nên bổ sung thêm kinh nghiệm hoặc học thêm về: Devops Engineer, It - Phần Mềm, GCP.
